# 8 · Transport: ions or quasi-particles ("_T")

| `transport` | |
|---|---|
| `ions` (default) | each total moves with one ion's diffusion coefficient and charge |
| `quasi` | each total moves with the sum of its species' Nernst–Planck fluxes (Herrmann & Horstmann 2024) |

Most Zn in 2 M ZnSO₄ is the neutral complex ZnSO₄⁰, which does not migrate, so the quasi-particle Zn total moves more slowly in the field than free Zn²⁺. The complexes' diffusion coefficients are not well known: `D_complex` (5×10⁻⁶ cm²/s by default) applies to every complex other than HSO₄⁻ and OH⁻.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model

def compare(runs, title, cols=("V", "pH_cath")):
    """Plot several runs (dict label -> Result) against capacity."""
    fig, ax = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4.2))
    ax = np.atleast_1d(ax)
    for label, r in runs.items():
        for a, c in zip(ax, cols):
            a.plot(r.column("mAhg"), r.column(c), label=label)
    for a, c in zip(ax, cols):
        a.set(xlabel="capacity [mAh/g]  (decreasing = charge)", ylabel=c)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def summary(runs):
    for label, r in runs.items():
        q = r.column("mAhg")
        print(f"{label:28s} exit {r.exit_reason:16s} discharged {q.max():6.1f} mAh/g, final {q[-1]:6.1f} mAh/g")

In [ ]:
runs = {}
for I in (100, 400):
    for t in ("ions", "quasi"):
        runs[f"{t}, {I} mA/g"] = run(Params(transport=t, steps=f"cc I={I} Vmin=1.0"))
summary(runs)
compare(runs, "transport", cols=("V", "pH_cath", "Zn_cath_M"));

Sensitivity to the complexes' diffusion coefficient (quasi-particle transport, 400 mA/g):

In [ ]:
runs = {f"D_complex = {d:g}": run(Params(transport="quasi", D_complex=d, steps="cc I=400 Vmin=1.0")) for d in (2e-6, 5e-6, 1e-5)}
summary(runs)
compare(runs, "complex diffusion coefficient", cols=("V", "Zn_cath_M"));